In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

In [3]:
from mtcnn import MTCNN

detector = MTCNN()

def extract_face(img_path, target_size=(160,160)):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(img_rgb)
    
    if results:
        x, y, w, h = results[0]['box']
        face = img_rgb[y:y+h, x:x+w]
        face = cv2.resize(face, target_size)
        return face
    else:
        return None

2025-10-30 07:36:01.532295: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
dataset_path = "../dataset/train/"   # Folder structure: dataset/person_name/imgs
persons = os.listdir(dataset_path)

faces = []
labels = []

for person in persons:
    person_path = os.path.join(dataset_path, person)
    for img_name in os.listdir(person_path):
        img_path = os.path.join(person_path, img_name)
        face = extract_face(img_path)
        if face is not None:
            faces.append(face)
            labels.append(person)

faces = np.asarray(faces)
labels = np.asarray(labels)
print(f"✅ Total Faces: {len(faces)}")

2025-10-30 07:36:38.158011: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [0,48,48,3] vs. [1,1,1,32]
2025-10-30 07:36:44.112446: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [0,24,24,3] vs. [1,1,1,28]


✅ Total Faces: 148


In [6]:
!pip install keras-facenet

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'keras-facenet' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'keras-facenet'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for keras-facenet: filename=keras_facenet-0.3.2-py3-none-any.whl size=10369 sha256=659d08f04b490b45a3c363d8a090c3ed3d06dccaae324ba3ba934bdd6afe1ce6
  Stored in directory: /home/shagor/.cache/pip/wheels/05/b0/f5/19ac49fedc10b1df3ee56b096edbcfa39d45794fccc6bcdbbf
Successfully built keras-facenet


In [ ]:
from keras_facenet import FaceNet

embedder = FaceNet()

def get_embedding(face_pixels):
    face_pixels = face_pixels.astype('float32')
    mean, std = face_pixels.mean(), face_pixels.std()
    face_pixels = (face_pixels - mean) / std
    sample = np.expand_dims(face_pixels, axis=0)
    yhat = embedder.embeddings(sample)
    return yhat[0]

In [ ]:
embeddings = []
for face in tqdm(faces):
    emb = get_embedding(face)
    embeddings.append(emb)

embeddings = np.asarray(embeddings)
print("✅ Embedding shape:", embeddings.shape)